# Occlusion Statistics Analysis

This notebook computes statistical analyses for the occlusion experiments described in the paper. The goal of these experiments is to evaluate whether multimodal models demonstrate task-aligned sensitivity to roof regions during roof-condition assessment.

For each image, the occlusion pipeline computes:

- **Roof Mean Difference (RMD):** the average embedding-distance change when roof-related cells are occluded.
- **Non-Roof Mean Difference (NRMD):** the average embedding-distance change when non-roof cells are occluded.

We then compute paired statistical comparisons between RMD and NRMD for each model. These analyses help determine whether occluding roof regions produces systematically larger response changes than occluding unrelated image regions.

The notebook reports:
- paired roof-minus-non-roof differences,
- paired t-tests,
- Wilcoxon signed-rank tests,
- effect sizes (Cohen's \(d_z\)),
- confidence intervals,
- and summary statistics.

The primary purpose of this analysis is not to establish semantic correctness or expert-level roof understanding, but rather to evaluate whether the models exhibit differential sensitivity to task-relevant image regions under occlusion.

In [1]:
import numpy as np
from scipy.stats import ttest_rel, wilcoxon, t
from math import sqrt

# -----------------------------
# GPT values
# -----------------------------

gpt_rmd = np.array([
    0.0856, 0.0920, 0.0730, 0.0689, 0.1127,
    0.0453, 0.0516, 0.0603, 0.1149, 0.0419,
    0.1128, 0.0785, 0.1465, 0.1508, 0.0822,
    0.0956, 0.1087, 0.2005, 0.0696, 0.1200
])

gpt_nrmd = np.array([
    0.0903, 0.0745, 0.0615, 0.0824, 0.0900,
    0.0375, 0.0485, 0.0900, 0.1000, 0.0442,
    0.1055, 0.0742, 0.1872, 0.1598, 0.0884,
    0.0930, 0.1230, 0.1898, 0.0768, 0.0959
])

# -----------------------------
# LLaVA values
# -----------------------------

llava_rmd = np.array([
    0.0033, 0.0324, 0.0949, 0.0093, 0.0262,
    0.0269, 0.0356, 0.0055, 0.0038, 0.0208,
    0.0709, 0.0085, 0.0007, 0.0000, 0.0340,
    0.0108, 0.0066, 0.0000, 0.0017, 0.0240
])

llava_nrmd = np.array([
    0.0057, 0.0545, 0.0723, 0.0059, 0.0200,
    0.0000, 0.0335, 0.0055, 0.0022, 0.0042,
    0.0313, 0.0062, 0.0000, 0.0000, 0.0321,
    0.0023, 0.0030, 0.0000, 0.0015, 0.0181
])


def paired_stats(name, rmd, nrmd):
    diff = rmd - nrmd

    n = len(diff)
    mean_diff = np.mean(diff)
    sd_diff = np.std(diff, ddof=1)

    # Paired t-test
    t_stat, t_p = ttest_rel(rmd, nrmd)

    # Wilcoxon signed-rank
    w_stat, w_p = wilcoxon(diff)

    # Cohen's dz
    cohens_dz = mean_diff / sd_diff

    # 95% confidence interval
    se = sd_diff / sqrt(n)
    t_crit = t.ppf(0.975, df=n-1)

    ci_low = mean_diff - t_crit * se
    ci_high = mean_diff + t_crit * se

    print("=" * 50)
    print(name)
    print("=" * 50)

    print("\nPaired differences:")
    print(diff)

    print(f"\nMean difference: {mean_diff:.6f}")
    print(f"SD difference: {sd_diff:.6f}")

    print("\nPaired t-test")
    print(f"t = {t_stat:.4f}")
    print(f"p = {t_p:.6f}")

    print("\nWilcoxon signed-rank")
    print(f"W = {w_stat}")
    print(f"p = {w_p:.6f}")

    print("\nEffect size")
    print(f"Cohen's dz = {cohens_dz:.4f}")

    print("\n95% Confidence Interval")
    print(f"[{ci_low:.6f}, {ci_high:.6f}]")
    print()


paired_stats("GPT", gpt_rmd, gpt_nrmd)
paired_stats("LLaVA", llava_rmd, llava_nrmd)

GPT

Paired differences:
[-0.0047  0.0175  0.0115 -0.0135  0.0227  0.0078  0.0031 -0.0297  0.0149
 -0.0023  0.0073  0.0043 -0.0407 -0.009  -0.0062  0.0026 -0.0143  0.0107
 -0.0072  0.0241]

Mean difference: -0.000055
SD difference: 0.016454

Paired t-test
t = -0.0149
p = 0.988229

Wilcoxon signed-rank
W = 95.0
p = 0.728506

Effect size
Cohen's dz = -0.0033

95% Confidence Interval
[-0.007756, 0.007646]

LLaVA

Paired differences:
[-0.0024 -0.0221  0.0226  0.0034  0.0062  0.0269  0.0021  0.      0.0016
  0.0166  0.0396  0.0023  0.0007  0.      0.0019  0.0085  0.0036  0.
  0.0002  0.0059]

Mean difference: 0.005880
SD difference: 0.012706

Paired t-test
t = 2.0697
p = 0.052360

Wilcoxon signed-rank
W = 21.0
p = 0.008607

Effect size
Cohen's dz = 0.4628

95% Confidence Interval
[-0.000066, 0.011826]

